# 01. 탐색적 데이터 분석 (EDA)

**Dataset**: `bigquery-public-data.google_analytics_sample.ga_sessions_*`  
**Period**: 2016-08-01 ~ 2017-08-01  
**Source**: Google Merchandise Store (실제 이커머스 트래픽)

## 목표
- 데이터 규모와 기간 파악
- 트래픽 트렌드와 계절성 확인
- 디바이스/채널/지역 분포 파악
- 이커머스 전환율 개요
- 사용자 참여도(세션당 페이지뷰) 분포

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from google.cloud import bigquery

# 스타일 설정
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

client = bigquery.Client()
print('BigQuery 연결 성공')

## 1.0 데이터 품질 점검 (Data Quality Check)

분석에 앞서 데이터의 무결성을 확인합니다:
- 주요 필드의 NULL 비율
- 날짜 범위 연속성 (누락 날짜 확인)
- 이상치 탐지 (세션 수 급변)

In [ ]:
# 1) 주요 필드 NULL 비율 점검
query_nulls = """
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(fullVisitorId IS NULL) AS null_visitor_id,
  COUNTIF(date IS NULL) AS null_date,
  COUNTIF(device.deviceCategory IS NULL) AS null_device,
  COUNTIF(channelGrouping IS NULL) AS null_channel,
  COUNTIF(geoNetwork.country IS NULL) AS null_country,
  COUNTIF(totals.pageviews IS NULL) AS null_pageviews,
  COUNTIF(totals.visits IS NULL) AS null_visits
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
"""

df_nulls = client.query(query_nulls).to_dataframe()
total = df_nulls['total_rows'].iloc[0]

print("=" * 50)
print("  DATA QUALITY CHECK: NULL 비율")
print("=" * 50)
for col in df_nulls.columns:
    if col == 'total_rows':
        continue
    null_count = df_nulls[col].iloc[0]
    pct = null_count / total * 100
    status = "✓ OK" if pct < 1 else f"⚠ {pct:.2f}%"
    field_name = col.replace('null_', '')
    print(f"  {field_name:<20} {status}")
print(f"\n총 {total:,}행 점검 완료")

In [ ]:
# 2) 날짜 연속성 점검 — 누락된 날짜가 있는지 확인
query_dates = """
SELECT PARSE_DATE('%Y%m%d', date) AS session_date
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY session_date
ORDER BY session_date
"""

df_dates = client.query(query_dates).to_dataframe()
df_dates['session_date'] = pd.to_datetime(df_dates['session_date'])

full_range = pd.date_range(df_dates['session_date'].min(), df_dates['session_date'].max())
missing = full_range.difference(df_dates['session_date'])

print(f"날짜 범위: {df_dates['session_date'].min().date()} ~ {df_dates['session_date'].max().date()}")
print(f"실제 존재 날짜: {len(df_dates)}일")
print(f"예상 날짜 수: {len(full_range)}일")
if len(missing) == 0:
    print("✓ 누락 날짜 없음 — 데이터 연속성 양호")
else:
    print(f"⚠ 누락 날짜 {len(missing)}일: {[d.strftime('%Y-%m-%d') for d in missing[:10]]}")

# 3) 이상치 탐지 — IQR 기반 일별 세션 수 급변 감지
query_daily_qc = """
SELECT PARSE_DATE('%Y%m%d', date) AS session_date, COUNT(*) AS sessions
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY session_date ORDER BY session_date
"""
df_daily_qc = client.query(query_daily_qc).to_dataframe()

q1 = df_daily_qc['sessions'].quantile(0.25)
q3 = df_daily_qc['sessions'].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df_daily_qc[(df_daily_qc['sessions'] < lower) | (df_daily_qc['sessions'] > upper)]

print(f"\n일별 세션 IQR: Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f}")
print(f"이상치 범위: < {lower:.0f} 또는 > {upper:.0f}")
print(f"이상치 일수: {len(outliers)}일 / {len(df_daily_qc)}일")
if len(outliers) > 0:
    print("이상치 날짜 (상위 5):")
    for _, r in outliers.nlargest(5, 'sessions').iterrows():
        print(f"  {r['session_date']}  {r['sessions']:,} sessions")

## 1.1 데이터 개요

In [ ]:
query_overview = """
SELECT
  COUNT(*) AS total_sessions,
  COUNT(DISTINCT fullVisitorId) AS unique_visitors,
  MIN(PARSE_DATE('%Y%m%d', date)) AS first_date,
  MAX(PARSE_DATE('%Y%m%d', date)) AS last_date,
  DATE_DIFF(
    MAX(PARSE_DATE('%Y%m%d', date)),
    MIN(PARSE_DATE('%Y%m%d', date)),
    DAY
  ) AS date_range_days
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE
  _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
"""

df_overview = client.query(query_overview).to_dataframe()
df_overview

In [ ]:
row = df_overview.iloc[0]
print(f"분석 기간: {row['first_date']} ~ {row['last_date']} ({row['date_range_days']}일)")
print(f"전체 세션 수: {row['total_sessions']:,}")
print(f"유니크 방문자 수: {row['unique_visitors']:,}")
print(f"방문자당 평균 세션: {row['total_sessions'] / row['unique_visitors']:.2f}")

## 1.2 일별 세션 추이

In [ ]:
query_daily = """
SELECT
  PARSE_DATE('%Y%m%d', date) AS session_date,
  COUNT(*) AS sessions,
  COUNT(DISTINCT fullVisitorId) AS visitors
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE
  _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY session_date
ORDER BY session_date
"""

df_daily = client.query(query_daily).to_dataframe()
df_daily['session_date'] = pd.to_datetime(df_daily['session_date'])
df_daily.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_daily['session_date'], df_daily['sessions'], linewidth=0.8, alpha=0.7, label='Daily Sessions')

# 7일 이동 평균
df_daily['sessions_ma7'] = df_daily['sessions'].rolling(7).mean()
ax.plot(df_daily['session_date'], df_daily['sessions_ma7'], linewidth=2, color='red', label='7-day MA')

ax.set_title('Daily Sessions Trend (2016-08 ~ 2017-08)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sessions')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## 1.3 디바이스 분포

In [ ]:
query_device = """
SELECT
  device.deviceCategory AS device,
  COUNT(*) AS sessions,
  COUNT(DISTINCT fullVisitorId) AS visitors,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS session_pct
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE
  _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY device
ORDER BY sessions DESC
"""

df_device = client.query(query_device).to_dataframe()
df_device

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors = ['#4e79a7', '#f28e2b', '#76b7b2']
axes[0].pie(
    df_device['sessions'], labels=df_device['device'],
    autopct='%1.1f%%', colors=colors, startangle=90
)
axes[0].set_title('Sessions by Device')

# Bar chart
axes[1].bar(df_device['device'], df_device['visitors'], color=colors)
axes[1].set_title('Unique Visitors by Device')
axes[1].set_ylabel('Visitors')
for i, v in enumerate(df_device['visitors']):
    axes[1].text(i, v + v*0.01, f'{v:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 1.4 채널별 세션 분포

In [ ]:
query_channel = """
SELECT
  channelGrouping AS channel,
  COUNT(*) AS sessions,
  COUNT(DISTINCT fullVisitorId) AS visitors,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS session_pct
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE
  _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY channel
ORDER BY sessions DESC
"""

df_channel = client.query(query_channel).to_dataframe()
df_channel

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_channel['channel'][::-1], df_channel['sessions'][::-1], color='#4e79a7')
ax.set_xlabel('Sessions')
ax.set_title('Sessions by Channel Grouping')

for bar, pct in zip(bars, df_channel['session_pct'][::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 1.5 국가별 분포 (Top 10)

In [ ]:
query_country = """
SELECT
  geoNetwork.country AS country,
  COUNT(*) AS sessions,
  COUNT(DISTINCT fullVisitorId) AS visitors,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS session_pct
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE
  _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY country
ORDER BY sessions DESC
LIMIT 10
"""

df_country = client.query(query_country).to_dataframe()
df_country

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(df_country['country'][::-1], df_country['sessions'][::-1], color='#59a14f')
ax.set_xlabel('Sessions')
ax.set_title('Top 10 Countries by Sessions')
plt.tight_layout()
plt.show()

## 1.6 이커머스 전환율

In [ ]:
query_ecom = """
SELECT
  COUNT(*) AS total_sessions,
  COUNTIF(totals.transactions > 0) AS purchase_sessions,
  ROUND(COUNTIF(totals.transactions > 0) * 100.0 / COUNT(*), 4) AS conversion_rate_pct,
  SUM(totals.totalTransactionRevenue) / 1e6 AS total_revenue_usd
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE
  _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
"""

df_ecom = client.query(query_ecom).to_dataframe()
row = df_ecom.iloc[0]
print(f"전체 세션: {row['total_sessions']:,.0f}")
print(f"구매 세션: {row['purchase_sessions']:,.0f}")
print(f"전환율: {row['conversion_rate_pct']:.4f}%")
print(f"총 매출: ${row['total_revenue_usd']:,.2f}")

## 1.7 세션당 페이지뷰 분포

In [ ]:
query_pageviews = """
SELECT
  totals.pageviews AS pageviews_per_session,
  COUNT(*) AS session_count
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE
  _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY pageviews_per_session
ORDER BY pageviews_per_session
LIMIT 30
"""

df_pv = client.query(query_pageviews).to_dataframe()
df_pv = df_pv.dropna(subset=['pageviews_per_session'])
df_pv['pageviews_per_session'] = df_pv['pageviews_per_session'].astype(int)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_pv['pageviews_per_session'], df_pv['session_count'], color='#e15759')
ax.set_xlabel('Pageviews per Session')
ax.set_ylabel('Number of Sessions')
ax.set_title('Distribution of Pageviews per Session')
ax.set_xlim(0, 30)
plt.tight_layout()
plt.show()

# 통계 요약
total = df_pv['session_count'].sum()
one_page = df_pv[df_pv['pageviews_per_session'] == 1]['session_count'].values[0]
print(f"1 페이지만 보고 이탈한 세션: {one_page:,} ({one_page/total*100:.1f}%)")

## 1.8 월별 트래픽 및 매출 추이

In [ ]:
query_monthly = """
SELECT
  FORMAT_DATE('%Y-%m', PARSE_DATE('%Y%m%d', date)) AS month,
  COUNT(*) AS sessions,
  COUNT(DISTINCT fullVisitorId) AS visitors,
  COUNTIF(totals.transactions > 0) AS purchases,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6, 2) AS revenue_usd
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE
  _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY month
ORDER BY month
"""

df_monthly = client.query(query_monthly).to_dataframe()
df_monthly

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(df_monthly['month'], df_monthly['sessions'], color='#4e79a7', alpha=0.7, label='Sessions')
ax1.set_xlabel('Month')
ax1.set_ylabel('Sessions', color='#4e79a7')
ax1.tick_params(axis='y', labelcolor='#4e79a7')
plt.xticks(rotation=45)

ax2 = ax1.twinx()
ax2.plot(df_monthly['month'], df_monthly['revenue_usd'], color='#e15759',
         marker='o', linewidth=2, label='Revenue (USD)')
ax2.set_ylabel('Revenue (USD)', color='#e15759')
ax2.tick_params(axis='y', labelcolor='#e15759')

fig.suptitle('Monthly Sessions & Revenue', fontsize=14)
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

In [ ]:
# Key Findings (데이터 기반 동적 생성)
overview = df_overview.iloc[0]
top_device = df_device.iloc[0]
top_channel = df_channel.iloc[0]
top_country = df_country.iloc[0]
ecom = df_ecom.iloc[0]
one_page_pct = one_page / total * 100

print("=" * 60)
print("  EDA KEY FINDINGS")
print("=" * 60)
print(f"""
1. 데이터 규모: {overview['total_sessions']:,}개 세션, {overview['unique_visitors']:,}명 유니크 방문자 ({overview['date_range_days']}일)

2. 디바이스: {top_device['device']}이 {top_device['session_pct']}%로 최대. 모바일 비중은 낮음

3. 채널: {top_channel['channel']}이 {top_channel['session_pct']}%로 최대 트래픽 소스

4. 지역: {top_country['country']}가 {top_country['session_pct']}%로 절대 다수 (Google Merchandise Store 특성)

5. 전환율: {ecom['conversion_rate_pct']:.4f}% (구매 세션 {ecom['purchase_sessions']:,.0f} / 전체 {ecom['total_sessions']:,.0f})
   총 매출: ${ecom['total_revenue_usd']:,.2f}

6. 참여도: {one_page_pct:.1f}%가 1페이지만 보고 이탈 (높은 바운스율)
""")
print("Next Steps:")
print("  → 퍼널 분석으로 구체적 이탈 지점 파악: 02_funnel.ipynb")
print("  → 리텐션 분석으로 재방문 패턴 파악: 03_retention.ipynb")